---
title : "Talk with WISKI"
execute:
    eval: false
    output: asis
    error: false
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

In de documentatie laten we voorbeelden zien hoe je verschillende tools gebruikt. 

In [1]:
# zorg dat alle variable goed staan
import json
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

## 1. Parameters

Met de tool `get_parameters` kan je de parameters van WISKI opvragen.

In [2]:
from talk_with_wiski.models import GetLocationsInput, GetParametersInput
from talk_with_wiski.tools import get_locations, get_parameters

Dit kan voor een aantal stations

In [3]:
get_parameter_input = GetParametersInput(station_no=["ZUN0034_KKL_OW", "0300_TDB"])
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.model_dump()

Of voor alle stations

In [4]:
import numpy as np

get_parameter_input = GetParametersInput()
get_parameter_output = await get_parameters(ctx, get_parameter_input)

data_dict = get_parameter_output.data
parameters = np.unique(
    [row["stationparameter_longname"].split(" [")[0].lower() for row in data_dict]
)
print(parameters)

In [5]:
get_parameter_input = GetParametersInput(
    station_no="BRE0301_PMP1",
)
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.data

## 2. Locations

Met de tool `get_locations` kan je de parameters van WISKI opvragen.

In [6]:
get_locations_input = GetLocationsInput(max_number_of_station_ids=5)

ctx = setup_dot_env()

get_locations_output = await get_locations(ctx, get_locations_input)

get_locations_output.model_dump()

### Using geocoding is also possible

This uses `from geopy.geocoders import Nominatim`, <br>
which uses the OSM API, [See the policy](https://operations.osmfoundation.org/policies/nominatim/)<br>
 -> `No heavy uses (an absolute maximum of 1 request per second).` 

In [7]:
get_locations_input = GetLocationsInput(
    requested_geolocation=["Loon op Zand", 5.0],
    returnfields="station_no,station_id,station_name,stationparameter_name,station_latitude,station_longitude,site_name",
)
print(input)
get_locations_output = await get_locations(ctx, input)
for item in get_locations_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_locations_output.data)}")

## 3. Sites

Met de tool `get_sites` kun je de beschikbare sites van WISKI opvragen.

Hier een aantal voorbeelden.

In [8]:
from talk_with_wiski.models import GetSitesInput
from talk_with_wiski.tools import get_sites

get_sites_input = GetSitesInput(
    returnfields="site_no,site_id,site_name",
)
get_sites_output = await get_sites(ctx, get_sites_input)
for item in get_sites_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_sites_output.data)}")

## 4. Group lists

In [9]:
from talk_with_wiski.models import GetGroupListInput, GetGroupListOutput
from talk_with_wiski.tools import get_group_list

# Test end time before start time
get_group_list_input = GetGroupListInput(group_type="timeseries")

get_group_list_output: GetGroupListOutput = await get_group_list(
    ctx, get_group_list_input
)
get_group_list_output.model_dump()

## 5. ParametertypeList

In [10]:
from talk_with_wiski.models import GetParametertypeListInput, GetParametertypeListOutput
from talk_with_wiski.tools import get_parametertype_list

In [11]:
print("Test ParametertypeList: get ids for given names")
get_parametertype_list_input = GetParametertypeListInput(
    parametertype_name=["Evaporation", "WNS*"],
)
print(get_parametertype_list_input)
get_parametertype_list_output: GetParametertypeListOutput = (
    await get_parametertype_list(ctx, get_parametertype_list_input)
)

print(f"Aantal parametertype gevonden {len(get_parametertype_list_output.data)}")
get_parametertype_list_output.model_dump()

## 6. Timeseries list

Met de tool `get_timeseries_list` kan je de beschikbare timeseries ophalen voor een locatie, site en/of een parameter.

In [12]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput
from talk_with_wiski.tools import get_timeseries_list

Hiervoor gebruiken we de eerste op gehaalde parameter en locatie combinatie: 

In [13]:
previously_retrieved_station = get_parameter_output.data[0]["station_no"]
previously_retrieved_parameter = [
    data["parametertype_name"] for data in get_parameter_output.data
]
previously_retrieved_station, previously_retrieved_parameter

In [14]:
get_timeseries_list_input = GetTimeseriesListInput(
    station_no=previously_retrieved_station,
    parametertype_name=previously_retrieved_parameter,
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

get_timeseries_list_output.model_dump()

In [15]:
get_timeseries_list_output.model_dump()["data"][0].keys()

## 7. Timeseries values

Met de tool `get_timeseries_values` kan je de timeseries data ophalen voor een timeseries_id.

In [16]:
from talk_with_wiski.models import GetTimeseriesValuesInput, GetTimeseriesValuesOutput
from talk_with_wiski.tools import get_timeseries_values

In [17]:
get_timeseries_value_input = GetTimeseriesValuesInput(
    ts_id=[item["ts_id"] for item in get_timeseries_list_output.data],
    from_date="2025-11-01",
    period="P1D",
    # to_date="2025-11-06"
)
get_timeseries_value_output: GetTimeseriesValuesOutput = await get_timeseries_values(
    ctx, get_timeseries_value_input
)

get_timeseries_value_output.model_dump()

In [18]:
[item["ts_id"] for item in get_timeseries_list_output.data]

## 8. Timeseries Type

Met de tool `get_timeseries_Type` kan je de informatie ophalen of het type tijdserie.

In [19]:
from talk_with_wiski.models import GetTimeseriesTypeInput, GetTimeseriesTypeOutput
from talk_with_wiski.tools import get_timeseries_type

We reuse the timeseries types from when we retrieved the timeseries list

In [20]:
ts_type_id = get_timeseries_list_output.data[0]["ts_type_id"]
ts_type_id

In [21]:
get_timeseries_type_input = GetTimeseriesTypeInput(ts_type_id=ts_type_id)
get_timeseries_type_output: GetTimeseriesTypeOutput = await get_timeseries_type(
    ctx, get_timeseries_type_input
)
get_timeseries_type_output.model_dump()